In [1]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import shutil
import torch

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

def locate_root(folder_name, marker):
    matches = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob(marker)
        if path.parent.name == folder_name
    })
    assert len(matches) == 1, (
        f"Expected one attached {folder_name}; found: {matches}"
    )
    return matches[0]

TINY_BASELINE_ROOT = locate_root(
    "HealthVer_TinyLlama_Experiments",
    "initialization_manifest.json",
)
ORIGINAL_TAGCOS_ROOT = locate_root(
    "HealthVer_TAGCOS_Experiments",
    "warmup_manifest.json",
)
TAGCOS_ROOT = Path(
    "/kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments"
)

required = [
    "input_config.json",
    "preprocessing_manifest.json",
    "dataset_metadata.json",
    "tokenized_train/state.json",
    "tokenizer/tokenizer_config.json",
    "initial_adapter_seed42/adapter_config.json",
    "initial_adapter_seed42/adapter_model.safetensors",
    "TinyLlama_LoRA_full_seed42/run_config.json",
    "TinyLlama_LoRA_full_seed42/training_complete.json",
]
for relative in required:
    assert (TINY_BASELINE_ROOT / relative).is_file(), relative

input_config = read_json(TINY_BASELINE_ROOT / "input_config.json")
preprocessing = read_json(
    TINY_BASELINE_ROOT / "preprocessing_manifest.json"
)
initialization = read_json(
    TINY_BASELINE_ROOT / "initialization_manifest.json"
)
baseline_config = read_json(
    TINY_BASELINE_ROOT / "TinyLlama_LoRA_full_seed42/run_config.json"
)

MODEL_ID = input_config["model"]
MODEL_REVISION = input_config["model_revision"]
DATASET_REVISION = input_config["dataset_revision"]
SEQUENCE_LIMIT = input_config["max_sequence_length"]
INITIAL_ADAPTER = TINY_BASELINE_ROOT / "initial_adapter_seed42"

assert MODEL_ID == "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
assert MODEL_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
assert DATASET_REVISION == "b20ac99ceed62f5264a319fa25a854df1668d85b"
assert SEQUENCE_LIMIT == preprocessing["sequence_limit"] == 448
assert input_config["truncation"] is False
assert input_config["loss"] == "assistant_answer_only"
assert initialization["optimizer_updates"] == 0
assert initialization["trainable_parameters"] == 12615680

assert sha256(
    INITIAL_ADAPTER / "adapter_model.safetensors"
) == initialization["initial_adapter_sha256"] == (
    baseline_config["initial_adapter_sha256"]
)
assert sha256(
    INITIAL_ADAPTER / "adapter_config.json"
) == baseline_config["initial_adapter_config_sha256"]
assert sha256(
    TINY_BASELINE_ROOT / "input_config.json"
) == baseline_config["input_config_sha256"]
assert sha256(
    TINY_BASELINE_ROOT / "preprocessing_manifest.json"
) == baseline_config["preprocessing_manifest_sha256"]

TRAIN_CSV = (
    TINY_BASELINE_ROOT / "original_data"
    / f"{DATASET_REVISION}_healthver_train.csv"
)
assert sha256(TRAIN_CSV) == (
    preprocessing["splits"]["train"]["source_csv_sha256"]
)

ORIGINAL_WARMUP_PATH = ORIGINAL_TAGCOS_ROOT / "warmup_manifest.json"
original_warmup = read_json(ORIGINAL_WARMUP_PATH)

assert original_warmup["method"] == "TAGCOS_HealthVer_adaptation"
assert original_warmup["sampling"] == "uniform_without_replacement"
assert original_warmup["selection_seed"] == 42
assert original_warmup["warmup_training_seed"] == 42
assert original_warmup["warmup_epochs"] == 3
assert original_warmup["checkpoint_policy"] == "each_epoch"
assert original_warmup["warmup_rows"] == 530

indices = original_warmup["row_indices"]
assert len(indices) == len(set(indices)) == 530
assert indices == sorted(indices)
assert all(type(i) is int and 0 <= i < 10590 for i in indices)
assert len(original_warmup["ids"]) == len(set(original_warmup["ids"])) == 530

print("TinyLlama baseline:", TINY_BASELINE_ROOT)
print("Original TAGCOS:", ORIGINAL_TAGCOS_ROOT)
print("Training CSV and initial-adapter hashes: PASSED")
print("Original warm-up manifest SHA256:", sha256(ORIGINAL_WARMUP_PATH))
print("Warm-up manifest structure and settings: PASSED")
print("Planned output:", TAGCOS_ROOT)

print("\nEnvironment:")
for package in [
    "torch", "transformers", "peft", "accelerate", "datasets",
    "scikit-learn", "scipy", "traker", "fast-jl", "torchao",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

assert torch.cuda.is_available(), "Enable the GPU accelerator."
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {properties.name}, "
        f"{properties.total_memory / 1024**3:.2f} GiB"
    )
print("CUDA compiler:", shutil.which("nvcc"))

print("\nCheck complete. No model loaded or files written.")
print("Validation and test contents not accessed.")

TinyLlama baseline: /kaggle/input/datasets/datasciencexlscout/base-3/HealthVer_TinyLlama_Experiments
Original TAGCOS: /kaggle/input/datasets/datasciencexlscout/tagcos-1/HealthVer_TAGCOS_Experiments
Training CSV and initial-adapter hashes: PASSED
Original warm-up manifest SHA256: c97c7499125d9bc270c73441606383b6d33b2f4f3d481b3b027c8ebe5cbf166d
Warm-up manifest structure and settings: PASSED
Planned output: /kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments

Environment:
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
datasets: 4.8.5
scikit-learn: 1.6.1
scipy: 1.16.3
traker: NOT INSTALLED
fast-jl: NOT INSTALLED
torchao: NOT INSTALLED
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB
CUDA compiler: /usr/local/cuda/bin/nvcc

Check complete. No model loaded or files written.
Validation and test contents not accessed.


In [3]:
import pandas as pd
from datasets import load_from_disk

full_train = load_from_disk(
    str(TINY_BASELINE_ROOT / "tokenized_train")
)
train_frame = pd.read_csv(
    TRAIN_CSV, dtype=str, keep_default_na=False
)
LABELS = ["Supports", "Refutes", "Neutral"]

assert len(full_train) == len(train_frame) == 10590
assert train_frame["id"].is_unique
assert set(train_frame["label"]).issubset(LABELS)
assert list(full_train["id"]) == train_frame["id"].tolist()

# Check actual saved contents, independently of the loaded cache fingerprint.
digest = hashlib.sha256()
for row in full_train:
    serialized = json.dumps(
        row, sort_keys=True, separators=(",", ":")
    ) + "\n"
    digest.update(serialized.encode("utf-8"))

training_content_sha256 = digest.hexdigest()
saved_train = preprocessing["splits"]["train"]
assert training_content_sha256 == saved_train["tokenized_content_sha256"]
assert training_content_sha256 == baseline_config["training_content_sha256"]
assert max(full_train["sequence_length"]) <= SEQUENCE_LIMIT == 448

saved_state = read_json(
    TINY_BASELINE_ROOT / "tokenized_train/state.json"
)
assert saved_state["_fingerprint"] == saved_train["dataset_fingerprint"]

# Reuse the original manifest's indices; do not sample again.
assert sha256(ORIGINAL_WARMUP_PATH) == (
    "c97c7499125d9bc270c73441606383b6d33"
    "b2f4f3d481b3b027c8ebe5cbf166d"
)
warmup_indices = original_warmup["row_indices"]
assert len(warmup_indices) == len(set(warmup_indices)) == 530
assert warmup_indices == sorted(warmup_indices)

warmup_data = full_train.select(warmup_indices)
warmup_ids = list(warmup_data["id"])
assert warmup_ids == original_warmup["ids"]
assert warmup_ids == train_frame.iloc[warmup_indices]["id"].tolist()

warmup_train = warmup_data.select_columns(
    ["input_ids", "attention_mask", "labels"]
)

warmup_manifest = {
    **original_warmup,
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "dataset_fingerprint": saved_train["dataset_fingerprint"],
    "tokenized_content_sha256": training_content_sha256,
    "source_csv_sha256": saved_train["source_csv_sha256"],
    "source_warmup_manifest_sha256": sha256(ORIGINAL_WARMUP_PATH),
    "initial_adapter_sha256": initialization["initial_adapter_sha256"],
    "sequence_limit": 448,
    "truncation": False,
    "row_indices": warmup_indices,
    "ids": warmup_ids,
}

manifest_path = TAGCOS_ROOT / "warmup_manifest.json"
if manifest_path.exists():
    assert read_json(manifest_path) == warmup_manifest, (
        "Existing warm-up manifest differs; inspect before continuing."
    )
else:
    TAGCOS_ROOT.mkdir(parents=True, exist_ok=True)
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(warmup_manifest, indent=2),
        encoding="utf-8",
    )
    temporary.replace(manifest_path)

print(f"Verified training rows: {len(full_train):,}")
print("Training tokenized-content hash: PASSED")
print("Warm-up rows:", len(warmup_train))
print("Exact Qwen warm-up indices and IDs: PASSED")
print("Planned warm-up epochs:", warmup_manifest["warmup_epochs"])
print("Saved:", manifest_path)
print("No model weights loaded or training performed.")
print("Validation and test contents not accessed.")

Verified training rows: 10,590
Training tokenized-content hash: PASSED
Warm-up rows: 530
Exact Qwen warm-up indices and IDs: PASSED
Planned warm-up epochs: 3
Saved: /kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments/warmup_manifest.json
No model weights loaded or training performed.
Validation and test contents not accessed.


In [4]:
import gc
import json
import math
from pathlib import Path
from importlib.metadata import version
import hashlib
import torch

from transformers import (
    AutoModelForCausalLM, Trainer, TrainingArguments,
    TrainerCallback, set_seed,
)
from peft import PeftModel
from torch.nn.utils.rnn import pad_sequence

DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)

WARMUP_DIR = TAGCOS_ROOT / "warmup_seed42"

input_config = json.loads(
    (TINY_BASELINE_ROOT / "input_config.json").read_text()
)
manifest = json.loads(
    (TAGCOS_ROOT / "warmup_manifest.json").read_text()
)
assert manifest["warmup_rows"] == len(warmup_train) == 530
assert manifest["warmup_epochs"] == 3
assert manifest["warmup_training_seed"] == manifest["selection_seed"] == 42
assert input_config["model"] == "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
assert input_config["model_revision"] == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
assert input_config["max_sequence_length"] == manifest["sequence_limit"] == 448
assert input_config["truncation"] is False
assert input_config["loss"] == "assistant_answer_only"
assert manifest["tokenized_content_sha256"] == training_content_sha256
assert list(full_train.select(manifest["row_indices"])["id"]) == manifest["ids"]
assert len(warmup_train) == len(warmup_data) == 530
for index in range(530):
    assert warmup_train[index] == {key: warmup_data[index][key]
                                 for key in ["input_ids", "attention_mask", "labels"]}
assert sha256(INITIAL_ADAPTER / "adapter_model.safetensors") == manifest["initial_adapter_sha256"]
assert sha256(INITIAL_ADAPTER / "adapter_config.json") == baseline_config["initial_adapter_config_sha256"]
assert initialization["optimizer_updates"] == 0
for package, expected in {
    "torch": "2.11.0+cu128", "transformers": "5.16.1", "peft": "0.20.0",
    "accelerate": "1.14.0", "datasets": "4.8.5", "scikit-learn": "1.6.1",
}.items():
    assert version(package) == expected, (package, version(package))


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(TINY_BASELINE_ROOT / "tokenizer"),
    local_files_only=True,
)
tokenizer.padding_side = "right"
assert tokenizer.pad_token_id is not None
assert hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() == baseline_config["chat_template_sha256"]

def training_collator(rows):
    padding = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    batch = {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding.items()
    }
    assert batch["input_ids"].shape[1] <= 448
    return batch

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class GradientCheckpointMetadata(TrainerCallback):
    def __init__(self):
        self.interval_lrs = []

    def on_step_begin(self, args, state, control, optimizer=None, **kwargs):
        # Learning rate used for this optimizer update.
        self.interval_lrs.append(float(optimizer.param_groups[0]["lr"]))

    def on_save(
        self, args, state, control,
        model=None, optimizer=None, **kwargs
    ):
        checkpoint = WARMUP_DIR / f"checkpoint-{state.global_step}"
        names_by_object = {
            id(parameter): name
            for name, parameter in model.named_parameters()
        }
        optimizer_state = optimizer.state_dict()
        parameter_names = {}

        for live_group, saved_group in zip(
            optimizer.param_groups, optimizer_state["param_groups"]
        ):
            assert len(live_group["params"]) == len(saved_group["params"])
            for parameter, saved_id in zip(
                live_group["params"], saved_group["params"]
            ):
                parameter_names[str(saved_id)] = names_by_object[id(parameter)]

        assert len(self.interval_lrs) == 17
        assert len(parameter_names) == sum(1 for p in model.parameters() if p.requires_grad)
        metadata = {
            "model": input_config["model"],
            "training_content_sha256": training_content_sha256,
            "initial_adapter_sha256": manifest["initial_adapter_sha256"],
            "step": int(state.global_step),
            "epoch": float(state.epoch),
            "mean_interval_learning_rate": (
                sum(self.interval_lrs) / len(self.interval_lrs)
            ),
            "interval_optimizer_steps": len(self.interval_lrs),
            "optimizer_parameter_names": parameter_names,
            "model_revision": input_config["model_revision"],
            "warmup_manifest_sha256": hashlib.sha256(
                (TAGCOS_ROOT / "warmup_manifest.json").read_bytes()
            ).hexdigest(),
        }
        (checkpoint / "gradient_metadata.json").write_text(
            json.dumps(metadata, indent=2)
        )
        self.interval_lrs.clear()


def verify_completed_warmup():
    # Never overwrite or silently restart an incomplete warmup.
    required_steps = (17, 34, 51)
    state_file = WARMUP_DIR / "trainer_state.json"
    args_file = WARMUP_DIR / "training_args.json"
    metrics_file = WARMUP_DIR / "train_results.json"
    for file in (state_file, args_file, metrics_file):
        assert file.is_file(), f"Incomplete warmup; inspect before resuming: {file}"
    state = json.loads(state_file.read_text())
    saved_args = json.loads(args_file.read_text())
    saved_metrics = json.loads(metrics_file.read_text())
    assert state["global_step"] == 51 and state["epoch"] >= 3 - 1e-6
    expected_args = {
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "learning_rate": 1e-4,
        "warmup_steps": 2,
        "lr_scheduler_type": "linear",
        "optim": "adamw_torch",
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "fp16": True,
        "bf16": False,
        "seed": 42,
        "data_seed": 42,
        "eval_strategy": "no",
        "save_strategy": "epoch",
    }
    for key, value in expected_args.items():
        assert saved_args[key] == value, f"Existing warmup configuration differs: {key}"
    assert saved_metrics["train_runtime"] > 0
    actual_steps = sorted(int(p.name.split("-")[-1]) for p in WARMUP_DIR.glob("checkpoint-*") if p.is_dir())
    assert actual_steps == list(required_steps)
    manifest_hash = hashlib.sha256((TAGCOS_ROOT / "warmup_manifest.json").read_bytes()).hexdigest()
    for step in required_steps:
        checkpoint = WARMUP_DIR / f"checkpoint-{step}"
        for filename in (
            "adapter_model.safetensors", "adapter_config.json", "optimizer.pt",
            "scheduler.pt", "trainer_state.json", "gradient_metadata.json",
        ):
            file = checkpoint / filename
            assert file.is_file() and file.stat().st_size > 0, file
        info = json.loads((checkpoint / "gradient_metadata.json").read_text())
        checkpoint_state = json.loads((checkpoint / "trainer_state.json").read_text())
        assert checkpoint_state["global_step"] == info["step"] == step
        assert info["epoch"] == step / 17
        assert info["interval_optimizer_steps"] == 17
        assert info["mean_interval_learning_rate"] > 0
        assert info["optimizer_parameter_names"]
        assert info["model_revision"] == input_config["model_revision"]
        assert info["warmup_manifest_sha256"] == manifest_hash
        assert info["training_content_sha256"] == training_content_sha256
        assert info["initial_adapter_sha256"] == manifest["initial_adapter_sha256"]
        print("Verified:", checkpoint.name)
    print("Verified completed warmup; reusing checkpoints 17, 34 and 51.")

if WARMUP_DIR.exists():
    verify_completed_warmup()
else:
    set_seed(42)
    warmup_base = AutoModelForCausalLM.from_pretrained(
        input_config["model"],
        revision=input_config["model_revision"],
        dtype=torch.float16,
        attn_implementation="sdpa",
    ).to(DEVICE)

    warmup_model = PeftModel.from_pretrained(
        warmup_base,
        str(TINY_BASELINE_ROOT / "initial_adapter_seed42"),
        is_trainable=True,
    )
    del warmup_base

    warmup_model.config.use_cache = False
    warmup_model.config.pad_token_id = tokenizer.pad_token_id
    warmup_model.gradient_checkpointing_disable()

    for name, parameter in warmup_model.named_parameters():
        if parameter.requires_grad:
            assert "lora_" in name
            parameter.data = parameter.data.float()
    del parameter

    assert sum(
        p.numel() for p in warmup_model.parameters() if p.requires_grad
    ) == 12615680

    steps_per_epoch = math.ceil(math.ceil(len(warmup_train) / 8) / 4)
    assert steps_per_epoch == 17

    warmup_args = SingleGPUArguments(
        output_dir=str(WARMUP_DIR),
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_steps=math.ceil(0.03 * 3 * steps_per_epoch),
        lr_scheduler_type="linear",
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        bf16=False,
        gradient_checkpointing=False,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=3,
        save_only_model=False,
        logging_steps=10,
        logging_first_step=True,
        report_to="none",
        dataloader_num_workers=0,
        seed=42,
        data_seed=42,
        label_names=["labels"],
    )

    assert warmup_args.world_size == 1
    assert warmup_args.device == DEVICE
    assert warmup_args.train_batch_size == 8

    warmup_trainer = Trainer(
        model=warmup_model,
        args=warmup_args,
        train_dataset=warmup_train,
        data_collator=training_collator,
        processing_class=tokenizer,
        callbacks=[GradientCheckpointMetadata()],
    )

    WARMUP_DIR.mkdir(parents=True, exist_ok=True)
    (WARMUP_DIR / "training_args.json").write_text(
        warmup_args.to_json_string()
    )

    print("Starting from:", INITIAL_ADAPTER)
    print("Expected optimizer steps:", 3 * steps_per_epoch)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    result = warmup_trainer.train()
    assert warmup_trainer.state.global_step == 51
    assert abs(warmup_trainer.state.epoch - 3.0) < 1e-6

    metrics = dict(result.metrics)
    metrics["peak_allocated_gib"] = (
        torch.cuda.max_memory_allocated(DEVICE) / 1024**3
    )
    warmup_trainer.save_metrics("train", metrics)
    warmup_trainer.save_state()

    checkpoints = sorted(
        WARMUP_DIR.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    assert len(checkpoints) == 3

    for checkpoint in checkpoints:
        for filename in [
            "adapter_model.safetensors", "adapter_config.json",
            "optimizer.pt", "scheduler.pt",
            "trainer_state.json", "gradient_metadata.json",
        ]:
            assert (checkpoint / filename).is_file(), (
                f"Missing: {checkpoint / filename}"
            )
        print("Verified checkpoint:", checkpoint.name)

    print(json.dumps(metrics, indent=2))
    print("Warmup complete. No validation or test evaluation performed.")

    verify_completed_warmup()
    del warmup_trainer, warmup_model
    gc.collect()
    torch.cuda.empty_cache()


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting from: /kaggle/input/datasets/datasciencexlscout/base-3/HealthVer_TinyLlama_Experiments/initial_adapter_seed42
Expected optimizer steps: 51


Step,Training Loss
1,8.030394
10,5.664930
20,0.302206
30,0.254233
40,0.250253
50,0.236812


Verified checkpoint: checkpoint-17
Verified checkpoint: checkpoint-34
Verified checkpoint: checkpoint-51
{
  "train_runtime": 175.8678,
  "train_samples_per_second": 9.041,
  "train_steps_per_second": 0.29,
  "total_flos": 2557941188788224.0,
  "train_loss": 1.3668742647358016,
  "epoch": 3.0,
  "peak_allocated_gib": 10.965085983276367
}
Warmup complete. No validation or test evaluation performed.
Verified: checkpoint-17
Verified: checkpoint-34
Verified: checkpoint-51
Verified completed warmup; reusing checkpoints 17, 34 and 51.


In [5]:
import os, sys, subprocess, importlib
import importlib.metadata as package_metadata
import torch

protected = ['torch', 'transformers', 'peft', 'accelerate', 'datasets',
             'scikit-learn', 'numpy', 'scipy']
before = {name: package_metadata.version(name) for name in protected}
required = {'torch':'2.11.0+cu128', 'transformers':'5.16.1', 'peft':'0.20.0',
            'accelerate':'1.14.0', 'datasets':'4.8.5', 'scikit-learn':'1.6.1',
            'scipy':'1.16.3'}
for name, expected in required.items():
    assert before[name] == expected, (name, before[name], expected)
assert torch.cuda.is_available()
assert torch.cuda.get_device_capability(0) == (7, 5), 'This installation targets the verified T4.'
needed = []
for name, expected in [('traker', '0.3.2'), ('fast-jl', '0.1.3')]:
    try:
        found = package_metadata.version(name)
        assert found == expected, (name, found, expected)
    except package_metadata.PackageNotFoundError:
        needed.append(f'{name}=={expected}')
if needed:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', *needed],
        env={**os.environ, 'MAX_JOBS':'2', 'TORCH_CUDA_ARCH_LIST':'7.5'},
    )
importlib.invalidate_caches()
assert {name: package_metadata.version(name) for name in protected} == before
from trak.projectors import CudaProjector, ProjectionType

def projection_smoke_test():
    device = torch.device('cuda:0')
    generator = torch.Generator(device=device).manual_seed(0)
    probe = torch.randn((16, 1024), device=device, dtype=torch.float16, generator=generator)
    projector = CudaProjector(grad_dim=1024, proj_dim=8192, seed=0,
                             proj_type=ProjectionType.rademacher, device=device, max_batch_size=16)
    with torch.no_grad():
        first = projector.project(probe, model_id=0).float().clone()
        second = projector.project(probe, model_id=0).float()
    assert first.shape == (16, 8192)
    assert torch.isfinite(first).all() and (first.norm(dim=1) > 0).all()
    assert torch.equal(first, second), 'Projection repeatability failed.'
projection_smoke_test()
torch.cuda.empty_cache()
print('traker:', package_metadata.version('traker'))
print('fast-jl:', package_metadata.version('fast-jl'))
print('Training package versions unchanged. CUDA projection smoke test: PASSED')


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for traker: filename=traker-0.3.2-py3-none-any.whl size=29027 sha256=790ed844c4140bb54ecdf9d4283339fd11ae589e43381636f1ff77ce81c56305
  Stored in directory: /root/.cache/pip/wheels/94/5d/b7/e2e6ee7a0793141139d942c5da6ef8dd532277a97fe31e8c7c
  Created wheel for fast-jl: filename=fast_jl-0.1.3-cp313-cp313-linux_x86_64.whl size=948385 sha256=0b5b1aaed34cdca067fbf3d6f34c73315de773286f9f74f584992ac4c59636d0
  Stored in directory: /root/.cache/pip/wheels/8e/be/a0/9f21db120dc77b1ef68768d6ef29db83f927792efe5c8a1a1e
Successfully built traker fast-jl
traker: 0.3.2
fast-jl: 0.1.3
Training package versions unchanged. CUDA projection smoke test: PASSED


In [6]:
import gc, os, json, math, time, hashlib
from pathlib import Path
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trak.projectors import CudaProjector, ProjectionType
from tqdm.auto import tqdm

TINY_BASELINE_ROOT = Path('/kaggle/input/datasets/datasciencexlscout/base-3/HealthVer_TinyLlama_Experiments')
BASELINE_ROOT = TINY_BASELINE_ROOT
TAGCOS_ROOT = Path('/kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments')
OUTPUT_DIR = TAGCOS_ROOT
WARMUP_DIR = TAGCOS_ROOT / 'warmup_seed42'
DEVICE = torch.device('cuda:0')
CHECKPOINT_STEPS = (17, 34, 51)
SUBSET_SIZES = {'50pct': 5295, '25pct': 2648, '10pct': 1059, '5pct': 530}
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def content_hash(dataset):
    digest = hashlib.sha256()
    for row in dataset:
        digest.update((json.dumps(row, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8'))
    return digest.hexdigest()

input_config = json.loads((TINY_BASELINE_ROOT / 'input_config.json').read_text())

WARMUP_MANIFEST = json.loads((TAGCOS_ROOT / 'warmup_manifest.json').read_text())
DATASET_FINGERPRINT = WARMUP_MANIFEST['dataset_fingerprint']

MODEL_ID = input_config['model']
MODEL_REVISION = input_config['model_revision']
DATASET_REVISION = input_config['dataset_revision']
assert MODEL_ID == 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
assert MODEL_REVISION == 'fe8a4ea1ffedaf415f4da2f062534de366a451e6'
assert DATASET_REVISION == 'b20ac99ceed62f5264a319fa25a854df1668d85b'

baseline_config = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/run_config.json').read_text())
baseline_completion = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/training_complete.json').read_text())
assert json_hash(baseline_config) == baseline_completion['run_config_sha256']
assert sha256(TINY_BASELINE_ROOT / 'input_config.json') == baseline_config['input_config_sha256']
for relative in ['tokenized_validation/state.json', 'preprocessing_manifest.json',
                 f'original_data/{DATASET_REVISION}_healthver_dev.csv',
                 f'original_data/{DATASET_REVISION}_healthver_test.csv']:
    assert (TINY_BASELINE_ROOT / relative).is_file(), f'Missing baseline artifact: {relative}'
source_paths = [p for p in Path('/kaggle/input').rglob('tagcos_protocol.json')
                if p.parent.name == 'HealthVer_TAGCOS_Experiments']
assert len(source_paths) == 1, 'Attach the original tagcos-1 dataset, including tagcos_protocol.json.'
source_protocol = json.loads(source_paths[0].read_text())
source_warmup_path = source_paths[0].parent / 'warmup_manifest.json'
assert sha256(source_warmup_path) == WARMUP_MANIFEST['source_warmup_manifest_sha256']
assert source_protocol['warmup_manifest_sha256'] == sha256(source_warmup_path)
assert input_config['max_sequence_length'] == WARMUP_MANIFEST['sequence_limit'] == 448
assert input_config['truncation'] is False
assert len(list(WARMUP_DIR.glob('checkpoint-*'))) == 3

def derive_parameter_order():
    # Derive the actual TinyLlama model parameter order.
    base = model = None
    try:
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
            attn_implementation='sdpa', device_map={'':'cuda:0'},
        )
        model = PeftModel.from_pretrained(base, str(WARMUP_DIR / 'checkpoint-17'), is_trainable=True)
        shapes = [(name, list(p.shape)) for name, p in model.named_parameters() if p.requires_grad]
        assert all('lora_' in name for name, shape in shapes)
        assert sum(math.prod(shape) for name, shape in shapes) == 12615680
        for step in CHECKPOINT_STEPS:
            metadata = json.loads((WARMUP_DIR / f'checkpoint-{step}' / 'gradient_metadata.json').read_text())
            assert set(metadata['optimizer_parameter_names'].values()) == {name for name, _ in shapes}
            assert metadata['warmup_manifest_sha256'] == sha256(TAGCOS_ROOT / 'warmup_manifest.json')
            assert metadata['training_content_sha256'] == WARMUP_MANIFEST['tokenized_content_sha256']
        return json_hash(shapes)
    finally:
        model = base = None
        gc.collect()
        torch.cuda.empty_cache()

parameter_order_sha256 = derive_parameter_order()
TAGCOS_CONFIG = {
    **source_protocol,
    'warmup_manifest_sha256':sha256(TAGCOS_ROOT / 'warmup_manifest.json'),
    'model':MODEL_ID, 'model_revision':MODEL_REVISION, 'dataset_revision':DATASET_REVISION,
    'dataset_fingerprint':DATASET_FINGERPRINT,
    'training_content_sha256':WARMUP_MANIFEST['tokenized_content_sha256'],
    'feature_dimensions':12615680, 'parameter_order_sha256':parameter_order_sha256,
    'source_protocol_sha256':sha256(source_paths[0]),
    'sequence_limit':448,
}
protocol_path = TAGCOS_ROOT / 'tagcos_protocol.json'
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == TAGCOS_CONFIG, 'Existing protocol differs.'
else:
    save_json(protocol_path, TAGCOS_CONFIG)
PROTOCOL_HASH = json_hash(TAGCOS_CONFIG)

assert TAGCOS_CONFIG['model'] == MODEL_ID and TAGCOS_CONFIG['model_revision'] == MODEL_REVISION
assert TAGCOS_CONFIG['dataset_revision'] == DATASET_REVISION
assert TAGCOS_CONFIG['dataset_fingerprint'] == DATASET_FINGERPRINT
assert TAGCOS_CONFIG['training_content_sha256'] == WARMUP_MANIFEST['tokenized_content_sha256']
assert TAGCOS_CONFIG['warmup_manifest_sha256'] == sha256(TAGCOS_ROOT / 'warmup_manifest.json')
assert TAGCOS_CONFIG['feature_dimensions'] == 12615680
PARAMETER_ORDER_SHA256 = TAGCOS_CONFIG['parameter_order_sha256']

assert TAGCOS_CONFIG['checkpoints'] == list(CHECKPOINT_STEPS)
assert TAGCOS_CONFIG['subset_sizes'] == SUBSET_SIZES

# All original algorithm fields except model-specific warm-up identity must match.
source_paths = [p for p in Path('/kaggle/input').rglob('tagcos_protocol.json')
                if p.parent.name == 'HealthVer_TAGCOS_Experiments']
assert len(source_paths) == 1
source_protocol = json.loads(source_paths[0].read_text())
assert sha256(source_paths[0]) == TAGCOS_CONFIG['source_protocol_sha256']
for key, value in source_protocol.items():
    if key != 'warmup_manifest_sha256':
        assert TAGCOS_CONFIG[key] == value, f'Original TAGCOS setting changed: {key}'
assert TAGCOS_CONFIG['reference_commit'] == '93a5a38681323e21aa9b6d503674bef814d7916c'
assert TAGCOS_CONFIG['adam_feature'] == '(0.9*m+0.1*g)/sqrt(0.999*v+0.001*g*g+1e-8)'
assert TAGCOS_CONFIG['projection_dim'] == 8192
assert TAGCOS_CONFIG['projection_seed'] == TAGCOS_CONFIG['projection_model_id'] == 0
assert TAGCOS_CONFIG['projection_batch_size'] == 16
assert TAGCOS_CONFIG['renormalize_after_aggregation'] is False

versions = {p: package_metadata.version(p) for p in (
    'torch', 'transformers', 'peft', 'accelerate', 'datasets',
    'scikit-learn', 'scipy', 'traker', 'fast-jl', 'numpy')}
for package, expected in {
    'torch':'2.11.0+cu128', 'transformers':'5.16.1', 'peft':'0.20.0',
    'accelerate':'1.14.0', 'datasets':'4.8.5', 'scikit-learn':'1.6.1',
}.items():
    assert versions[package] == expected, (package, versions[package], expected)
for key, package in [('scikit_learn', 'scikit-learn'), ('scipy', 'scipy'),
                     ('traker', 'traker'), ('fast_jl', 'fast-jl')]:
    assert TAGCOS_CONFIG[key] == versions[package]
save_json(TAGCOS_ROOT / 'continuation_environment.json', versions)

full_train = load_from_disk(str(TINY_BASELINE_ROOT / 'tokenized_train'))
assert len(full_train) == 10590
train_state = json.loads((TINY_BASELINE_ROOT / 'tokenized_train/state.json').read_text())
assert train_state['_fingerprint'] == DATASET_FINGERPRINT
assert content_hash(full_train) == WARMUP_MANIFEST['tokenized_content_sha256']
tokenizer = AutoTokenizer.from_pretrained(str(TINY_BASELINE_ROOT / 'tokenizer'), local_files_only=True)
assert hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() == baseline_config['chat_template_sha256']
assert sha256(TINY_BASELINE_ROOT / 'initial_adapter_seed42/adapter_model.safetensors') == WARMUP_MANIFEST['initial_adapter_sha256']

for step in CHECKPOINT_STEPS:
    checkpoint = WARMUP_DIR / f'checkpoint-{step}'
    for filename in ['adapter_config.json', 'adapter_model.safetensors', 'optimizer.pt', 'gradient_metadata.json']:
        assert (checkpoint / filename).is_file()
    info = json.loads((checkpoint / 'gradient_metadata.json').read_text())
    assert info['step'] == step and info['interval_optimizer_steps'] == 17
    assert abs(info['epoch'] - step / 17) < 1e-6
    assert info['model_revision'] == MODEL_REVISION
    assert info['warmup_manifest_sha256'] == TAGCOS_CONFIG['warmup_manifest_sha256']
    assert info['mean_interval_learning_rate'] > 0

def feature_collator(row):
    return {key: torch.tensor([row[key]], dtype=torch.long, device=DEVICE)
            for key in ('input_ids', 'attention_mask', 'labels')}

def inspect_features(path, unit_norm=True):
    array = np.load(path, mmap_mode='r', allow_pickle=False)
    assert array.shape == (len(full_train), 8192) and array.dtype == np.float32
    for start in range(0, len(array), 512):
        block = np.asarray(array[start:start + 512])
        assert np.isfinite(block).all()
        if unit_norm:
            assert np.allclose(np.linalg.norm(block, axis=1), 1.0, rtol=1e-4, atol=1e-5)
    del array

def checkpoint_feature_identity(step):
    checkpoint = WARMUP_DIR / f'checkpoint-{step}'
    return {
        'protocol_sha256': PROTOCOL_HASH,
        'dataset_fingerprint': DATASET_FINGERPRINT,
        'training_content_sha256': WARMUP_MANIFEST['tokenized_content_sha256'],
        'checkpoint_step': step,
        'model_revision': MODEL_REVISION,
        'parameter_order_sha256': PARAMETER_ORDER_SHA256,
        'adapter_sha256': sha256(checkpoint / 'adapter_model.safetensors'),
        'optimizer_sha256': sha256(checkpoint / 'optimizer.pt'),
        'gradient_metadata_sha256': sha256(checkpoint / 'gradient_metadata.json'),
    }

print('Protocol and all three warm-up checkpoints: PASSED')
print('Actual TinyLlama parameter-order SHA256:', PARAMETER_ORDER_SHA256)
print('Original selection settings preserved. Validation/test contents not accessed.')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Protocol and all three warm-up checkpoints: PASSED
Actual TinyLlama parameter-order SHA256: 5ae14e884d96953ddf370e209150b49689bc27fce3ae1087d57a91f996ccb45b
Original selection settings preserved. Validation/test contents not accessed.


In [7]:
# Trainer steps count attempted updates; Adam counts executed updates.
def check_update_counters(trainer_step, adam_steps, scheduler_step,
                          previous_trainer_step, previous_adam_step):
    assert adam_steps, 'No Adam state counters found.'
    assert all(math.isfinite(value) and value.is_integer() for value in adam_steps)
    unique_steps = {int(value) for value in adam_steps}
    assert len(unique_steps) == 1, 'Adam counters disagree across adapter tensors.'
    actual = unique_steps.pop()
    assert 0 < actual <= trainer_step, 'Invalid Adam update count.'
    assert scheduler_step == actual, 'Scheduler and Adam update counts disagree.'
    attempted_interval = trainer_step - previous_trainer_step
    completed_interval = actual - previous_adam_step
    assert attempted_interval == 17
    assert 0 < completed_interval <= attempted_interval
    return {
        'trainer_global_step':trainer_step,
        'adam_updates':actual,
        'attempted_updates_in_interval':attempted_interval,
        'completed_updates_in_interval':completed_interval,
        'skipped_updates_in_interval':attempted_interval-completed_interval,
        'skipped_updates_cumulative':trainer_step-actual,
    }

def audit_warmup_optimizer_counters():
    reports = {}
    previous_trainer_step = previous_adam_step = 0
    for step in CHECKPOINT_STEPS:
        checkpoint = WARMUP_DIR / f'checkpoint-{step}'
        trainer_state = json.loads((checkpoint / 'trainer_state.json').read_text())
        metadata = json.loads((checkpoint / 'gradient_metadata.json').read_text())
        assert trainer_state['global_step'] == metadata['step'] == step
        assert abs(trainer_state['epoch'] - step/17) < 1e-6
        assert metadata['interval_optimizer_steps'] == 17
        assert metadata['warmup_manifest_sha256'] == TAGCOS_CONFIG['warmup_manifest_sha256']
        snapshot = torch.load(checkpoint / 'optimizer.pt', map_location='cpu', weights_only=True)
        scheduler = torch.load(checkpoint / 'scheduler.pt', map_location='cpu', weights_only=True)
        scaler = torch.load(checkpoint / 'scaler.pt', map_location='cpu', weights_only=True)
        mapped_ids = {int(key) for key in metadata['optimizer_parameter_names']}
        grouped_ids = [pid for group in snapshot['param_groups'] for pid in group['params']]
        assert set(grouped_ids) == set(snapshot['state']) == mapped_ids
        assert len(grouped_ids) == len(set(grouped_ids)) == len(mapped_ids) == 308
        for group in snapshot['param_groups']:
            assert tuple(group['betas']) == (0.9, 0.999) and group['eps'] == 1e-8
        counters = []
        element_count = 0
        for state in snapshot['state'].values():
            counters.append(float(state['step']))
            m, v = state['exp_avg'], state['exp_avg_sq']
            assert m.shape == v.shape
            assert torch.isfinite(m).all() and torch.isfinite(v).all() and (v >= 0).all()
            element_count += m.numel()
        assert element_count == 12615680
        record = check_update_counters(step, counters, scheduler['last_epoch'],
                                       previous_trainer_step, previous_adam_step)
        assert math.isfinite(float(scaler['scale'])) and float(scaler['scale']) > 0
        record.update({
            'checkpoint':checkpoint.name,
            'optimizer_state_tensors':len(mapped_ids),
            'scheduler_last_epoch':scheduler['last_epoch'],
            'gradient_scale':float(scaler['scale']),
            'moment_elements':element_count,
            'optimizer_sha256':sha256(checkpoint / 'optimizer.pt'),
            'scheduler_sha256':sha256(checkpoint / 'scheduler.pt'),
            'scaler_sha256':sha256(checkpoint / 'scaler.pt'),
            'gradient_metadata_sha256':sha256(checkpoint / 'gradient_metadata.json'),
            'mean_interval_learning_rate':metadata['mean_interval_learning_rate'],
        })
        reports[str(step)] = record
        previous_trainer_step, previous_adam_step = step, record['adam_updates']
        print(f"{checkpoint.name}: {step} attempted / {record['adam_updates']} executed Adam updates; "
              f"{record['skipped_updates_in_interval']} skipped in this interval. States verified.")
        del snapshot, scheduler, scaler, state, m, v
        gc.collect()
    report = {
        'protocol_sha256':PROTOCOL_HASH,
        'checkpoints':reports,
        'learning_rate_weighting_note':
            'Preserves the original recorded mean LR over attempted Trainer updates, including skipped attempts. '
            'The legacy interval_optimizer_steps metadata field counts attempts, not necessarily executed Adam updates.',
        'correction':'Validation and reporting only; checkpoints, moments, feature formula and LR weights unchanged.',
    }
    destination = TAGCOS_ROOT / 'warmup_optimizer_audit.json'
    if destination.exists():
        assert json.loads(destination.read_text()) == report, 'Saved warm-up audit differs.'
    else:
        save_json(destination, report)
    return reports

WARMUP_OPTIMIZER_AUDIT = audit_warmup_optimizer_counters()

def extract_checkpoint(step):
    checkpoint = WARMUP_DIR / f'checkpoint-{step}'
    directory = TAGCOS_ROOT / 'features' / checkpoint.name
    directory.mkdir(parents=True, exist_ok=True)
    final = directory / 'normalized_features.npy'
    meta_path = directory / 'feature_metadata.json'
    identity_path = directory / 'feature_identity.json'
    partial = directory / 'normalized_features.resume.npy'
    progress_path = directory / 'extraction_progress.json'
    identity = checkpoint_feature_identity(step)
    checkpoint_meta = json.loads((checkpoint / 'gradient_metadata.json').read_text())

    if final.exists():
        assert meta_path.exists(), f'Array without completion metadata: {directory}'
        saved = json.loads(meta_path.read_text())
        expected = {'rows':len(full_train),'dimensions':8192,'projection_seed':0,
                    'projection_model_id':0,'projection_batch_size':16,
                    'projection':'CudaProjector_rademacher',
                    'projection_input_dtype':'float16','storage_dtype':'float32',
                    'normalization':'per_checkpoint_row_L2',
                    'row_order':'original_tokenized_train_order',
                    'dataset_fingerprint':DATASET_FINGERPRINT}
        for k,v in expected.items():
            assert saved[k] == v, (directory,k)
        assert saved['protocol_sha256'] == PROTOCOL_HASH
        assert saved['parameter_order_sha256'] == PARAMETER_ORDER_SHA256
        assert Path(saved['checkpoint']).name == checkpoint.name
        assert saved['mean_interval_learning_rate'] == checkpoint_meta['mean_interval_learning_rate']
        inspect_features(final)
        assert identity_path.exists(), f'Missing feature identity: {directory}'
        receipt = json.loads(identity_path.read_text())
        assert receipt['identity'] == identity
        assert receipt['features_sha256'] == sha256(final)
        print(f'{checkpoint.name}: verified completed features; skipped.')
        return

    progress = None
    if progress_path.exists():
        progress = json.loads(progress_path.read_text())
        assert progress['identity'] == identity, 'Partial extraction configuration changed.'
        assert partial.exists() and not meta_path.exists()
    else:
        assert not meta_path.exists() and not identity_path.exists(), 'Incomplete completion metadata.'
        assert not partial.exists(), 'Partial array lacks progress metadata; retain files and inspect.'
    done = int(progress['completed_rows']) if progress else 0
    previous_seconds = float(progress['extraction_seconds']) if progress else 0.0
    assert 0 <= done <= len(full_train)
    assert done % 16 == 0 or done == len(full_train)
    mmap = np.lib.format.open_memmap(partial, mode='r+' if progress else 'w+',
        dtype=np.float32, shape=(len(full_train),8192))
    assert mmap.shape == (len(full_train),8192) and mmap.dtype == np.float32
    if progress is None:
        mmap.flush()
        save_json(progress_path, {'identity': identity, 'completed_rows': 0,
                                 'extraction_seconds': 0.0, 'parameter_order_sha256': PARAMETER_ORDER_SHA256})
    if done:
        assert np.isfinite(mmap[:done]).all()
        assert np.allclose(np.linalg.norm(mmap[:done],axis=1),1,rtol=1e-4,atol=1e-5)

    base = model = moments = projector = buf = None
    m = v = g = u = batch = loss = projected = norms = None
    params = []
    started = None
    try:
        base = AutoModelForCausalLM.from_pretrained(
            input_config['model'],revision=input_config['model_revision'],
            dtype=torch.float16,attn_implementation='sdpa').to(DEVICE)
        model = PeftModel.from_pretrained(base, str(checkpoint), is_trainable=True)
        model.config.use_cache = False
        model.config.pad_token_id = tokenizer.pad_token_id
        model.gradient_checkpointing_disable()
        model.eval()
        params = [(name,p) for name,p in model.named_parameters() if p.requires_grad]
        for name,p in params:
            assert 'lora_' in name
            p.data = p.data.float()
        del p
        assert sum(p.numel() for _,p in params) == 12615680
        name_to_id = {name:int(sid) for sid,name in checkpoint_meta['optimizer_parameter_names'].items()}
        assert set(name_to_id) == {name for name,_ in params}
        assert identity['optimizer_sha256'] == WARMUP_OPTIMIZER_AUDIT[str(step)]['optimizer_sha256']
        moments = torch.load(checkpoint/'optimizer.pt',map_location='cpu',weights_only=True)
        for group in moments['param_groups']:
            assert tuple(group['betas']) == (0.9,0.999) and group['eps'] == 1e-8
        first,second = [],[]
        for name,p in params:
            state = moments['state'][name_to_id[name]]
            assert float(state['step']) == WARMUP_OPTIMIZER_AUDIT[str(step)]['adam_updates']
            assert state['exp_avg'].shape == state['exp_avg_sq'].shape == p.shape
            first.append(state['exp_avg'].float().flatten())
            second.append(state['exp_avg_sq'].float().flatten())
        m = torch.cat(first).to(DEVICE)
        v = torch.cat(second).to(DEVICE)
        assert torch.isfinite(m).all() and torch.isfinite(v).all() and (v >= 0).all()
        del first,second,state,p
        moments = None
        parameter_order_sha256 = json_hash([(name,list(p.shape)) for name,p in params])
        assert parameter_order_sha256 == PARAMETER_ORDER_SHA256
        projector = CudaProjector(grad_dim=m.numel(),proj_dim=8192,seed=0,
            proj_type=ProjectionType.rademacher,device=DEVICE,max_batch_size=16)
        buf = torch.empty((16,m.numel()),dtype=torch.float16,device=DEVICE)
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        started = time.perf_counter()

        for start in tqdm(range(done,len(full_train),16),desc=f'{checkpoint.name} features'):
            stop = min(start+16,len(full_train))
            buf.zero_()
            for pos,index in enumerate(range(start,stop)):
                model.zero_grad(set_to_none=True)
                batch = feature_collator(full_train[index])
                assert batch['input_ids'].shape[1] <= 448
                assert (batch['labels'] != -100).any()
                with torch.enable_grad(), torch.autocast('cuda',dtype=torch.float16):
                    loss = model(**batch).loss
                assert torch.isfinite(loss), index
                (loss * 1024.0).backward()
                assert all(p.grad is not None for _,p in params)
                g = torch.cat([p.grad.detach().float().flatten() for _,p in params]) / 1024.0
                assert torch.isfinite(g).all(), index
                u = (0.9*m + 0.1*g) / torch.sqrt(0.999*v + 0.001*g.square() + 1e-8)
                assert torch.isfinite(u).all(), index
                if step == 17 and start == 0:
                    assert float(g.norm()) > 0 and float(u.norm()) > 0
                    if pos < 3:
                        print({'row_index':index, 'loss':float(loss.detach()),
                               'gradient_norm':float(g.norm()), 'adam_feature_norm':float(u.norm()),
                               'feature_dimensions':u.numel()})
                buf[pos].copy_(u)
                del g,u,loss,batch
            model.zero_grad(set_to_none=True)
            assert torch.isfinite(buf).all()
            with torch.no_grad():
                projected = projector.project(buf,model_id=0)[:stop-start].float()
                assert torch.isfinite(projected).all()
                norms = projected.norm(dim=1,keepdim=True)
                assert torch.isfinite(norms).all() and (norms > 0).all()
                normalized = projected / norms
                assert torch.allclose(normalized.norm(dim=1), torch.ones(stop-start, device=DEVICE), rtol=1e-4, atol=1e-5)
                if step == 17 and start == 0:
                    repeated = projector.project(buf, model_id=0)[:stop-start].float()
                    assert torch.equal(projected, repeated), 'Real-feature projection repeatability failed.'
                    torch.cuda.synchronize(DEVICE)
                    elapsed_probe = time.perf_counter() - started
                    sanity = {'examples':stop-start, 'input_dimensions':12615680,
                              'projected_dimensions':8192, 'seconds':elapsed_probe,
                              'rough_three_checkpoint_minutes':elapsed_probe/(stop-start)*len(full_train)*3/60,
                              'repeatability':'PASSED', 'unit_norm_check':'PASSED',
                              'protocol_sha256':PROTOCOL_HASH}
                    save_json(TAGCOS_ROOT / 'feature_sanity.json', sanity)
                    print(json.dumps(sanity, indent=2))
                    del repeated
                mmap[start:stop] = normalized.cpu().numpy()
                del normalized
            del projected,norms
            done = stop
            if done % 256 == 0 or done == len(full_train):
                mmap.flush()
                save_json(progress_path,{'identity':identity,'completed_rows':done,
                    'extraction_seconds':previous_seconds+time.perf_counter()-started,
                    'parameter_order_sha256':parameter_order_sha256})
        torch.cuda.synchronize(DEVICE)
        elapsed = previous_seconds + time.perf_counter() - started
        mmap.flush()
        del mmap
        inspect_features(partial)
        report = {
            'checkpoint':str(checkpoint),'rows':len(full_train),'dimensions':8192,
            'trainer_global_step':step,
            'executed_adam_updates':WARMUP_OPTIMIZER_AUDIT[str(step)]['adam_updates'],
            'skipped_updates_cumulative':WARMUP_OPTIMIZER_AUDIT[str(step)]['skipped_updates_cumulative'],
            'lr_weight_basis':'saved_mean_over_attempted_trainer_updates',
            'projection_seed':0,'projection_model_id':0,'projection_batch_size':16,
            'projection':'CudaProjector_rademacher','projection_input_dtype':'float16',
            'storage_dtype':'float32','normalization':'per_checkpoint_row_L2',
            'row_order':'original_tokenized_train_order',
            'dataset_fingerprint':DATASET_FINGERPRINT,
            'mean_interval_learning_rate':checkpoint_meta['mean_interval_learning_rate'],
            'extraction_seconds':elapsed,
            'peak_allocated_gib':torch.cuda.max_memory_allocated(DEVICE)/1024**3,
            'protocol_sha256':PROTOCOL_HASH,'parameter_order_sha256':parameter_order_sha256,
            'input_dimensions':12615680,'fingerprint_source':'persisted_dataset_state',
            'training_content_sha256':WARMUP_MANIFEST['tokenized_content_sha256'],
            'runtime_note':'Excludes model setup and unsaved interrupted work.',
        }
        os.replace(partial,final)
        save_json(meta_path,report)
        save_json(identity_path,{'identity':identity,'features_sha256':sha256(final),
            'provenance':'continuation_extraction'})
        progress_path.unlink(missing_ok=True)
        print(json.dumps(report,indent=2))
    finally:
        if 'mmap' in locals():
            mmap.flush()
            del mmap
        params.clear()
        base = model = moments = projector = buf = None
        # Clear local GPU tensors even when an assertion interrupts extraction.
        m = v = g = u = batch = loss = projected = norms = None
        gc.collect()
        torch.cuda.empty_cache()

for checkpoint_step in CHECKPOINT_STEPS:
    extract_checkpoint(checkpoint_step)
    gc.collect()
    torch.cuda.empty_cache()
print('All three checkpoint feature files complete.')




checkpoint-17: 17 attempted / 12 executed Adam updates; 5 skipped in this interval. States verified.
checkpoint-34: 34 attempted / 29 executed Adam updates; 0 skipped in this interval. States verified.
checkpoint-51: 51 attempted / 46 executed Adam updates; 0 skipped in this interval. States verified.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

checkpoint-17 features:   0%|          | 0/662 [00:00<?, ?it/s]

{'row_index': 0, 'loss': 0.38033464550971985, 'gradient_norm': 8.51211166381836, 'adam_feature_norm': 3096.72119140625, 'feature_dimensions': 12615680}
{'row_index': 1, 'loss': 0.21281297504901886, 'gradient_norm': 4.598931312561035, 'adam_feature_norm': 2259.29443359375, 'feature_dimensions': 12615680}
{'row_index': 2, 'loss': 0.28570619225502014, 'gradient_norm': 5.984770774841309, 'adam_feature_norm': 2927.11962890625, 'feature_dimensions': 12615680}
{
  "examples": 16,
  "input_dimensions": 12615680,
  "projected_dimensions": 8192,
  "seconds": 3.8553458639999008,
  "rough_three_checkpoint_minutes": 127.5878521867467,
  "repeatability": "PASSED",
  "unit_norm_check": "PASSED",
  "protocol_sha256": "4479743573babc0356e6d0b524f97731f14862c9239e57e679e4e298943b4d35"
}
{
  "checkpoint": "/kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments/warmup_seed42/checkpoint-17",
  "rows": 10590,
  "dimensions": 8192,
  "trainer_global_step": 17,
  "executed_adam_updates": 12,
  "skipped_update

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

checkpoint-34 features:   0%|          | 0/662 [00:00<?, ?it/s]

{
  "checkpoint": "/kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments/warmup_seed42/checkpoint-34",
  "rows": 10590,
  "dimensions": 8192,
  "trainer_global_step": 34,
  "executed_adam_updates": 29,
  "skipped_updates_cumulative": 5,
  "lr_weight_basis": "saved_mean_over_attempted_trainer_updates",
  "projection_seed": 0,
  "projection_model_id": 0,
  "projection_batch_size": 16,
  "projection": "CudaProjector_rademacher",
  "projection_input_dtype": "float16",
  "storage_dtype": "float32",
  "normalization": "per_checkpoint_row_L2",
  "row_order": "original_tokenized_train_order",
  "dataset_fingerprint": "f403508492ed587a",
  "mean_interval_learning_rate": 6.326530612244898e-05,
  "extraction_seconds": 2249.7480667769996,
  "peak_allocated_gib": 4.161019325256348,
  "protocol_sha256": "4479743573babc0356e6d0b524f97731f14862c9239e57e679e4e298943b4d35",
  "parameter_order_sha256": "5ae14e884d96953ddf370e209150b49689bc27fce3ae1087d57a91f996ccb45b",
  "input_dimensions": 12615680,
  

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

checkpoint-51 features:   0%|          | 0/662 [00:00<?, ?it/s]

{
  "checkpoint": "/kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments/warmup_seed42/checkpoint-51",
  "rows": 10590,
  "dimensions": 8192,
  "trainer_global_step": 51,
  "executed_adam_updates": 46,
  "skipped_updates_cumulative": 5,
  "lr_weight_basis": "saved_mean_over_attempted_trainer_updates",
  "projection_seed": 0,
  "projection_model_id": 0,
  "projection_batch_size": 16,
  "projection": "CudaProjector_rademacher",
  "projection_input_dtype": "float16",
  "storage_dtype": "float32",
  "normalization": "per_checkpoint_row_L2",
  "row_order": "original_tokenized_train_order",
  "dataset_fingerprint": "f403508492ed587a",
  "mean_interval_learning_rate": 2.857142857142857e-05,
  "extraction_seconds": 2228.421973724,
  "peak_allocated_gib": 4.161046028137207,
  "protocol_sha256": "4479743573babc0356e6d0b524f97731f14862c9239e57e679e4e298943b4d35",
  "parameter_order_sha256": "5ae14e884d96953ddf370e209150b49689bc27fce3ae1087d57a91f996ccb45b",
  "input_dimensions": 12615680,
  "fin

In [8]:
feature_paths = [TAGCOS_ROOT/'features'/f'checkpoint-{s}'/'normalized_features.npy'
                 for s in CHECKPOINT_STEPS]
feature_reports = [json.loads(p.with_name('feature_metadata.json').read_text()) for p in feature_paths]
learning_rates = np.array([r['mean_interval_learning_rate'] for r in feature_reports],dtype=np.float64)
assert np.isfinite(learning_rates).all() and (learning_rates > 0).all()
checkpoint_weights = learning_rates / learning_rates.sum()
aggregate_path = TAGCOS_ROOT / 'aggregated_features.npy'
aggregate_meta_path = TAGCOS_ROOT / 'aggregation_metadata.json'
aggregation_identity = {
    'protocol_sha256':PROTOCOL_HASH,'dataset_fingerprint':DATASET_FINGERPRINT,
    'input_sha256':[sha256(p) for p in feature_paths],
    'checkpoint_steps':list(CHECKPOINT_STEPS),
    'checkpoint_weights':checkpoint_weights.tolist(),
}
if aggregate_meta_path.exists():
    aggregation_metadata = json.loads(aggregate_meta_path.read_text())
    assert aggregation_metadata['identity'] == aggregation_identity
    assert aggregation_metadata['sha256'] == sha256(aggregate_path)
    inspect_features(aggregate_path,unit_norm=False)
    print('Verified existing aggregation.')
else:
    assert not aggregate_path.exists(), 'Aggregation array lacks completion metadata.'
    started = time.perf_counter()
    arrays = [np.load(p,mmap_mode='r',allow_pickle=False) for p in feature_paths]
    tmp = aggregate_path.with_name('aggregated_features.partial.npy')
    out = np.lib.format.open_memmap(tmp,mode='w+',dtype=np.float32,shape=(len(full_train),8192))
    for start in range(0,len(full_train),512):
        stop = min(start+512,len(full_train))
        block = np.zeros((stop-start,8192),dtype=np.float32)
        for weight,arr in zip(checkpoint_weights,arrays):
            block += np.float32(weight) * arr[start:stop]
        assert np.isfinite(block).all()
        out[start:stop] = block
    out.flush()
    del out,arrays,block
    os.replace(tmp,aggregate_path)
    aggregation_metadata = {'identity':aggregation_identity,'sha256':sha256(aggregate_path),
                            'aggregation_seconds':time.perf_counter()-started}
    save_json(aggregate_meta_path,aggregation_metadata)
print('Checkpoint weights:', dict(zip(CHECKPOINT_STEPS,checkpoint_weights.tolist())))
print('Saved:',aggregate_path)




Checkpoint weights: {17: 0.464473223661183, 34: 0.36891844592229606, 51: 0.1666083304165208}
Saved: /kaggle/working/HealthVer_TinyLlama_TAGCOS_Experiments/aggregated_features.npy


In [9]:
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits
from scipy.linalg import lstsq

def allocate_quotas(cluster_counts, total):
    counts = np.asarray(cluster_counts,dtype=np.int64)
    assert (counts >= 0).all() and 0 < total <= counts.sum()
    numerators = counts * int(total)
    quotas = numerators // counts.sum()
    remainders = numerators % counts.sum()
    order = np.lexsort((np.arange(len(counts)), -remainders))
    for cluster_id in order[:int(total-quotas.sum())]:
        quotas[cluster_id] += 1
    assert quotas.sum() == total and (quotas <= counts).all()
    return quotas

def positive_omp_round(x, target, budget, lam=1.0, tol=1e-4):
    # x: examples x feature dimensions, NumPy float32 on CPU.
    assert x.ndim == 2 and target.shape == (x.shape[1],)
    assert 0 < budget <= len(x)
    target_norm = float(np.linalg.norm(target))
    if not np.isfinite(target_norm) or target_norm <= 1e-12:
        raise RuntimeError('Degenerate cluster centroid; inspect before changing the method.')
    residual = target.copy()
    active = []
    weights = np.empty(0,dtype=x.dtype)
    removed = 0
    for _ in range(budget):
        if float(np.linalg.norm(residual)) / target_norm < tol:
            break
        scores = x @ residual
        index = int(np.argmax(scores))
        # Preserve the reference's stop-on-reselected-atom behavior.
        if index in active or float(scores[index]) <= 0:
            break
        active.append(index)
        if len(active) == 1:
            denom = np.dot(x[index],x[index])
            assert float(denom) > 0
            weights = (scores[index] / denom).reshape(1)
        else:
            while active:
                atoms = x[active]
                gram = atoms @ atoms.T + np.float32(lam) * np.eye(len(active),dtype=x.dtype)
                weights = lstsq(gram,atoms @ target,lapack_driver='gelsy')[0]
                assert np.isfinite(weights).all()
                if float(weights.min()) >= 0:
                    break
                active.pop(int(np.argmin(weights)))
                removed += 1
            if not active:
                return [], removed
        residual = target - x[active].T @ weights
    # The upstream selector retains only nonzero coefficients.
    retained = [i for i,w in zip(active,weights.tolist()) if w > 0]
    return retained, removed

def select_cluster(x, target, quota):
    if quota == 0:
        return [], {'rounds':0,'removed_negative_coefficients':0}
    remaining = np.arange(len(x),dtype=np.int64)
    selected, rounds, removed = [], 0, 0
    while len(selected) < quota:
        need = quota - len(selected)
        support,dropped = positive_omp_round(x[remaining],target,need,
            lam=TAGCOS_CONFIG['omp']['lambda'],tol=TAGCOS_CONFIG['omp']['tol'])
        if not support:
            raise RuntimeError('OMP made no progress before reaching its quota; no fallback applied.')
        chosen = remaining[np.asarray(support,dtype=np.int64)]
        selected.extend(chosen.tolist())
        keep = np.ones(len(remaining),dtype=bool)
        keep[np.asarray(support,dtype=np.int64)] = False
        remaining = remaining[keep]
        rounds += 1
        removed += dropped
    assert len(selected) == len(set(selected)) == quota
    return selected, {'rounds':rounds,'removed_negative_coefficients':removed}

cluster_dir = TAGCOS_ROOT / 'clusters'
cluster_dir.mkdir(exist_ok=True)
cluster_file = cluster_dir / 'kmeans_clusters.npz'
cluster_meta_file = cluster_dir / 'cluster_metadata.json'
cluster_identity = {'protocol_sha256':PROTOCOL_HASH,
                    'aggregated_features_sha256':sha256(aggregate_path)}
X = np.load(aggregate_path,mmap_mode='r',allow_pickle=False)
assert X.shape == (len(full_train),8192) and np.isfinite(X).all()
if cluster_meta_file.exists():
    cluster_metadata = json.loads(cluster_meta_file.read_text())
    assert cluster_metadata['identity'] == cluster_identity
    assert cluster_metadata['sha256'] == sha256(cluster_file)
else:
    assert not cluster_file.exists(), 'Cluster file lacks completion metadata.'
    started = time.perf_counter()
    with threadpool_limits(limits=4):
        fitted = KMeans(**TAGCOS_CONFIG['kmeans']).fit(np.array(X,copy=True))
    np.savez(cluster_file,labels=fitted.labels_.astype(np.int64),centers=fitted.cluster_centers_.astype(np.float32))
    cluster_metadata = {'identity':cluster_identity,'sha256':sha256(cluster_file),
                        'clustering_seconds':time.perf_counter()-started,
                        'inertia':float(fitted.inertia_),'iterations':int(fitted.n_iter_)}
    save_json(cluster_meta_file,cluster_metadata)
    del fitted
with np.load(cluster_file,allow_pickle=False) as clustered:
    cluster_labels = clustered['labels'].copy()
    cluster_centers = clustered['centers'].copy()
assert cluster_labels.shape == (len(full_train),)
assert cluster_centers.shape == (100,8192) and np.isfinite(cluster_centers).all()
assert (cluster_labels >= 0).all() and (cluster_labels < 100).all()
counts = np.bincount(cluster_labels,minlength=100)
assert (counts > 0).all(), 'An empty cluster needs inspection.'
print('Cluster size range:',int(counts.min()),'to',int(counts.max()))

selections_dir = TAGCOS_ROOT / 'selections'
selections_dir.mkdir(exist_ok=True)
train_metadata = json.loads((BASELINE_ROOT/'dataset_metadata.json').read_text())
train_csv = BASELINE_ROOT/'original_data'/f"{train_metadata['revision']}_healthver_train.csv"
assert sha256(train_csv) == train_metadata['files']['train']['sha256']
train_frame = pd.read_csv(train_csv,dtype=str,keep_default_na=False)
assert train_frame['id'].tolist() == [str(i) for i in full_train['id']]
assert set(train_frame['label']).issubset(input_config['labels'])
summary = []
old_threads = torch.get_num_threads()
torch.set_num_threads(min(4,os.cpu_count() or 1))
try:
    for subset_name,size in SUBSET_SIZES.items():
        manifest_path = selections_dir / f'subset_{subset_name}.json'
        quotas = allocate_quotas(counts,size)
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            assert manifest['protocol'] == TAGCOS_CONFIG
            assert manifest['aggregated_features_sha256'] == sha256(aggregate_path)
            assert manifest['clusters_sha256'] == sha256(cluster_file)
            assert manifest['cluster_quotas'] == quotas.tolist()
            print(subset_name, ': reusing saved selection')
        else:
            started = time.perf_counter()
            selected_rows, cluster_trace = [], []
            with threadpool_limits(limits=4):
                for c in tqdm(range(100),desc=f'OMP {subset_name}'):
                    row_ids = np.flatnonzero(cluster_labels == c)
                    if quotas[c]:
                        values = np.array(X[row_ids],dtype=np.float32,copy=True)
                        target = cluster_centers[c].copy()
                        local_indices,trace = select_cluster(values,target,int(quotas[c]))
                        selected_rows.extend(row_ids[local_indices].tolist())
                        del values,target
                    else:
                        trace = {'rounds':0,'removed_negative_coefficients':0}
                    cluster_trace.append({'cluster':c,'cluster_size':int(counts[c]),
                                          'quota':int(quotas[c]),**trace})
            indices = sorted(selected_rows)
            assert len(indices) == len(set(indices)) == size
            manifest = {'method':TAGCOS_CONFIG['method'],'subset':subset_name,
                'protocol':TAGCOS_CONFIG,'protocol_sha256':PROTOCOL_HASH,
                'dataset_fingerprint':DATASET_FINGERPRINT,
                'aggregated_features_sha256':sha256(aggregate_path),
                'clusters_sha256':sha256(cluster_file),
                'row_indices':indices,'ids':[str(i) for i in full_train.select(indices)['id']],
                'cluster_quotas':quotas.tolist(),'cluster_trace':cluster_trace,
                'selection_seconds':time.perf_counter()-started,
                'training_weights':'uniform'}
            save_json(manifest_path,manifest)
        indices = manifest['row_indices']
        assert len(indices) == len(set(indices)) == size and indices == sorted(indices)
        assert all(0 <= i < len(full_train) for i in indices)
        assert manifest['dataset_fingerprint'] == DATASET_FINGERPRINT
        assert manifest['ids'] == [str(i) for i in full_train.select(indices)['id']]
        assert np.array_equal(np.bincount(cluster_labels[indices],minlength=100),quotas)
        class_counts = train_frame.iloc[indices]['label'].value_counts()
        summary.append({'subset':subset_name,'rows':size,
                        **{label:int(class_counts.get(label,0)) for label in input_config['labels']}})
finally:
    torch.set_num_threads(old_threads)
del X
selection_summary = pd.DataFrame(summary)
selection_summary.to_csv(TAGCOS_ROOT/'selection_summary.csv',index=False)
print(selection_summary.to_string(index=False))
print('All selections fixed. Validation and test were not used for selection.')




Cluster size range: 9 to 279


OMP 50pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 25pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 10pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 5pct:   0%|          | 0/100 [00:00<?, ?it/s]

subset  rows  Supports  Refutes  Neutral
 50pct  5295      1891     1205     2199
 25pct  2648       946      603     1099
 10pct  1059       377      242      440
  5pct   530       189      120      221
All selections fixed. Validation and test were not used for selection.


In [10]:
SELECTION_DIR = TAGCOS_ROOT / 'selections'
TRAIN_SUBSETS, SUBSET_MANIFESTS = {}, {}
for name, size in SUBSET_SIZES.items():
    selection_manifest = json.loads((SELECTION_DIR / f'subset_{name}.json').read_text())
    assert selection_manifest['protocol'] == TAGCOS_CONFIG
    assert selection_manifest['protocol_sha256'] == PROTOCOL_HASH
    assert selection_manifest['dataset_fingerprint'] == DATASET_FINGERPRINT
    assert selection_manifest['aggregated_features_sha256'] == sha256(aggregate_path)
    assert selection_manifest['clusters_sha256'] == sha256(cluster_file)
    indices = selection_manifest['row_indices']
    assert len(indices) == len(set(indices)) == size and indices == sorted(indices)
    selected = full_train.select(indices)
    assert list(selected['id']) == selection_manifest['ids']
    TRAIN_SUBSETS[name] = selected
    SUBSET_MANIFESTS[name] = selection_manifest
    print(f'{name}: {len(selected):,} training rows ready')

# Selection is now fixed; load the shared validation split for checkpoint selection.
full_validation = load_from_disk(str(TINY_BASELINE_ROOT / 'tokenized_validation'))
preprocessing = json.loads((TINY_BASELINE_ROOT / 'preprocessing_manifest.json').read_text())
validation_state = json.loads((TINY_BASELINE_ROOT / 'tokenized_validation/state.json').read_text())
VALIDATION_SAVED_FINGERPRINT = validation_state['_fingerprint']
assert VALIDATION_SAVED_FINGERPRINT == preprocessing['splits']['validation']['dataset_fingerprint']
dataset_metadata = json.loads((BASELINE_ROOT / 'dataset_metadata.json').read_text())
validation_csv = BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_healthver_dev.csv'
assert sha256(validation_csv) == dataset_metadata['files']['validation']['sha256']
validation_frame = pd.read_csv(validation_csv, dtype=str, keep_default_na=False)
assert len(full_validation) == len(validation_frame) == 1917
for raw, saved in zip(validation_frame.itertuples(index=False), full_validation):
    messages = [
        {'role':'system', 'content':input_config['system_prompt']},
        {'role':'user', 'content':f'Claim: {raw.claim}\n\nEvidence: {raw.evidence}'},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)
    ids = tokenizer.apply_chat_template(messages + [{'role':'assistant','content':raw.label}],
                                       tokenize=True, add_generation_prompt=False, return_dict=False)
    assert ids[:len(prompt)] == prompt and len(ids) <= 448
    assert saved == {'id':str(raw.id), 'input_ids':ids, 'attention_mask':[1]*len(ids),
                     'labels':[-100]*len(prompt)+ids[len(prompt):], 'sequence_length':len(ids)}
VALIDATION_CONTENT_SHA256 = content_hash(full_validation)
assert VALIDATION_CONTENT_SHA256 == preprocessing['splits']['validation']['tokenized_content_sha256']
assert VALIDATION_CONTENT_SHA256 == baseline_config['validation_content_sha256']
save_json(TAGCOS_ROOT / 'training_data_identity.json', {
    'training_content_sha256':WARMUP_MANIFEST['tokenized_content_sha256'],
    'validation_content_sha256':VALIDATION_CONTENT_SHA256,
    'saved_training_fingerprint':DATASET_FINGERPRINT,
    'saved_validation_fingerprint':VALIDATION_SAVED_FINGERPRINT,
    'selection_manifest_sha256':{name:sha256(SELECTION_DIR/f'subset_{name}.json') for name in SUBSET_SIZES},
})
print('Shared validation:', len(full_validation))
print('All training subsets and validation contents verified. Test data not accessed.')


50pct: 5,295 training rows ready
25pct: 2,648 training rows ready
10pct: 1,059 training rows ready
5pct: 530 training rows ready
Shared validation: 1917
All training subsets and validation contents verified. Test data not accessed.


In [11]:
# Train and test the four frozen TinyLlama TAGCOS subsets.
import gc
import json
import math
import time
import hashlib
from pathlib import Path
from importlib.metadata import version

import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
    Trainer, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

assert all(name in globals() for name in [
    'TINY_BASELINE_ROOT', 'OUTPUT_DIR', 'SELECTION_DIR',
    'full_train', 'full_validation',
]), 'Run the preflight and corrected subset-preparation cells first.'
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def content_hash(dataset):
    h = hashlib.sha256()
    for row in dataset:
        h.update((json.dumps(row, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8'))
    return h.hexdigest()

input_config = json.loads((TINY_BASELINE_ROOT / 'input_config.json').read_text())
baseline_config = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/run_config.json').read_text())
baseline_completion = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/training_complete.json').read_text())
dataset_metadata = json.loads((TINY_BASELINE_ROOT / 'dataset_metadata.json').read_text())
provenance = {
    'model':input_config['model'], 'model_revision':input_config['model_revision'],
    'dataset_revision':input_config['dataset_revision'], 'sequence_limit':448, 'truncation':False,
    'initial_adapter_sha256':WARMUP_MANIFEST['initial_adapter_sha256'],
    'verified_content_sha256':{'train':WARMUP_MANIFEST['tokenized_content_sha256'],
                               'validation':VALIDATION_CONTENT_SHA256},
    'selection_manifest_sha256':{name:sha256(SELECTION_DIR/f'subset_{name}.json') for name in SUBSET_SIZES},
}
preprocessing = json.loads((TINY_BASELINE_ROOT / 'preprocessing_manifest.json').read_text())
assert sha256(TINY_BASELINE_ROOT / 'preprocessing_manifest.json') == baseline_config['preprocessing_manifest_sha256']
assert json_hash(baseline_config) == baseline_completion['run_config_sha256']
assert provenance['model'] == input_config['model']
assert provenance['model_revision'] == input_config['model_revision']
assert provenance['dataset_revision'] == input_config['dataset_revision']
assert provenance['sequence_limit'] == 448 and provenance['truncation'] is False
for split in ['train', 'validation']:
    assert provenance['verified_content_sha256'][split] == preprocessing['splits'][split]['tokenized_content_sha256']
assert sha256(TINY_BASELINE_ROOT / 'input_config.json') == baseline_config['input_config_sha256']

MODEL_ID = input_config['model']
MODEL_REVISION = input_config['model_revision']
DATASET_REVISION = input_config['dataset_revision']
LABELS = input_config['labels']
SEQUENCE_LIMIT = input_config['max_sequence_length']
assert MODEL_ID == 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
assert MODEL_REVISION == 'fe8a4ea1ffedaf415f4da2f062534de366a451e6'
assert DATASET_REVISION == dataset_metadata['revision'] == 'b20ac99ceed62f5264a319fa25a854df1668d85b'
assert LABELS == ['Supports', 'Refutes', 'Neutral']
assert SEQUENCE_LIMIT == 448 and input_config['truncation'] is False
assert input_config['loss'] == 'assistant_answer_only'
assert (TINY_BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_healthver_test.csv').is_file(), 'Saved baseline is missing the original test CSV.'

SEED = baseline_config['seed']
EPOCHS = baseline_config['epochs']
MICRO_BATCH = baseline_config['micro_batch_size']
GRAD_ACCUM = baseline_config['gradient_accumulation']
assert (SEED, EPOCHS, MICRO_BATCH, GRAD_ACCUM) == (42, 3, 8, 4)
assert baseline_config['learning_rate'] == 1e-4
for key, value in [('optimizer', 'adamw_torch'), ('weight_decay', 0.01),
                   ('max_grad_norm', 1.0), ('scheduler', 'linear')]:
    assert baseline_config[key] == value

package_versions = {name: version(name) for name in [
    'torch', 'transformers', 'peft', 'accelerate', 'datasets', 'scikit-learn',
]}
assert package_versions == {
    'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'peft': '0.20.0',
    'accelerate': '1.14.0', 'datasets': '4.8.5', 'scikit-learn': '1.6.1',
}, package_versions

INITIAL_ADAPTER = TINY_BASELINE_ROOT / 'initial_adapter_seed42'
initialization = json.loads((TINY_BASELINE_ROOT / 'initialization_manifest.json').read_text())
initial_hashes = {name: sha256(INITIAL_ADAPTER / name) for name in [
    'adapter_config.json', 'adapter_model.safetensors',
]}
assert initial_hashes['adapter_model.safetensors'] == initialization['initial_adapter_sha256']
assert initial_hashes['adapter_model.safetensors'] == baseline_config['initial_adapter_sha256']
assert initial_hashes['adapter_config.json'] == baseline_config['initial_adapter_config_sha256']
assert initial_hashes['adapter_model.safetensors'] == provenance['initial_adapter_sha256']
assert initialization['optimizer_updates'] == 0
assert (baseline_config['lora_rank'], baseline_config['lora_alpha'], baseline_config['lora_dropout']) == (16, 32, 0.05)
assert baseline_config['target_modules'] == 'all-linear'
assert baseline_config['attention_implementation'] == 'sdpa'
assert baseline_config['gradient_checkpointing'] is False
assert baseline_config['evaluation_batch_size'] == 8
assert baseline_config['selection_metric'] == 'validation_macro_f1'
assert initialization['trainable_parameters'] == 12615680

tokenizer = AutoTokenizer.from_pretrained(str(TINY_BASELINE_ROOT / 'tokenizer'), local_files_only=True)
assert tokenizer.pad_token_id is not None
assert hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() == baseline_config['chat_template_sha256']
generation_config = GenerationConfig.from_dict(baseline_config['generation_config'])
assert generation_config.max_new_tokens == 12
assert generation_config.do_sample is False and generation_config.num_beams == 1
assert generation_config.pad_token_id == tokenizer.pad_token_id
assert generation_config.eos_token_id == tokenizer.eos_token_id

def load_original(split, filename):
    path = TINY_BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_{filename}'
    assert sha256(path) == dataset_metadata['files'][split]['sha256']
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert frame['id'].is_unique and set(frame['label']).issubset(LABELS)
    return frame

train_frame = load_original('train', 'healthver_train.csv')
validation_frame = load_original('validation', 'healthver_dev.csv')
assert len(train_frame) == len(full_train) == 10590
assert len(validation_frame) == len(full_validation) == 1917
assert list(full_train['id']) == train_frame['id'].tolist()
assert list(full_validation['id']) == validation_frame['id'].tolist()
assert set(train_frame['id']).isdisjoint(validation_frame['id'])
# Stable content checks replace the incorrect loaded-cache-fingerprint check.
assert content_hash(full_train) == provenance['verified_content_sha256']['train']
assert content_hash(full_validation) == provenance['verified_content_sha256']['validation']

SUBSET_SIZES = {'50pct': 5295, '25pct': 2648, '10pct': 1059, '5pct': 530}
TRAIN_SUBSETS = {}
subset_content_hashes = {}
for name, size in SUBSET_SIZES.items():
    path = SELECTION_DIR / f'subset_{name}.json'
    assert sha256(path) == provenance['selection_manifest_sha256'][name]
    manifest = json.loads(path.read_text())
    assert manifest['method'] == TAGCOS_CONFIG['method']
    assert manifest['protocol'] == TAGCOS_CONFIG and manifest['protocol_sha256'] == PROTOCOL_HASH
    assert manifest['dataset_fingerprint'] == DATASET_FINGERPRINT
    assert manifest['aggregated_features_sha256'] == sha256(aggregate_path)
    assert manifest['clusters_sha256'] == sha256(cluster_file)
    indices = manifest['row_indices']
    assert len(indices) == len(set(indices)) == size
    assert all(type(i) is int and 0 <= i < len(full_train) for i in indices)
    assert indices == sorted(indices)
    selected = full_train.select(indices)
    assert list(selected['id']) == manifest['ids'] == train_frame.iloc[indices]['id'].tolist()
    subset_content_hashes[name] = content_hash(selected)
    TRAIN_SUBSETS[name] = selected

def make_messages(claim, evidence):
    return [
        {'role': 'system', 'content': input_config['system_prompt']},
        {'role': 'user', 'content': f'Claim: {claim}\n\nEvidence: {evidence}'},
    ]

label_lookup = {label.casefold(): label for label in LABELS}

def parse_prediction(response):
    return label_lookup.get(' '.join(response.strip().split()).casefold(), 'INVALID')

def training_collator(rows):
    padding = {'input_ids': tokenizer.pad_token_id, 'attention_mask': 0, 'labels': -100}
    batch = {
        key: pad_sequence([torch.tensor(row[key], dtype=torch.long) for row in rows],
                          batch_first=True, padding_value=value)
        for key, value in padding.items()
    }
    assert batch['input_ids'].shape[1] <= SEQUENCE_LIMIT
    return batch

def evaluate_labels(eval_model, frame, folder, split, fingerprint):
    folder.mkdir(parents=True, exist_ok=True)
    previous_padding, previous_mode = tokenizer.padding_side, eval_model.training
    records = []
    started = time.perf_counter()
    max_prompt_tokens = 0
    try:
        tokenizer.padding_side = 'left'
        eval_model.eval()
        with torch.inference_mode():
            for start in tqdm(range(0, len(frame), 8), desc=f'{folder.parent.name}: {split}', leave=False):
                rows = frame.iloc[start:start + 8]
                prompts = [tokenizer.apply_chat_template(
                    make_messages(row.claim, row.evidence), tokenize=False, add_generation_prompt=True,
                ) for row in rows.itertuples(index=False)]
                inputs = tokenizer(prompts, padding=True, truncation=False,
                                   add_special_tokens=False, return_tensors='pt').to(DEVICE)
                width = inputs['input_ids'].shape[1]
                max_prompt_tokens = max(max_prompt_tokens, width)
                limit = SEQUENCE_LIMIT if split == 'validation' else eval_model.config.max_position_embeddings
                assert width + generation_config.max_new_tokens <= limit
                generated = eval_model.generate(**inputs, generation_config=generation_config)
                responses = tokenizer.batch_decode(generated[:, width:], skip_special_tokens=True)
                for uid, gold, response in zip(rows['id'], rows['label'], responses):
                    records.append({'id': uid, 'true_label': gold,
                                    'predicted_label': parse_prediction(response), 'raw_response': response})
                del inputs, generated
        assert [r['id'] for r in records] == frame['id'].tolist()
        gold, predicted = [r['true_label'] for r in records], [r['predicted_label'] for r in records]
        metrics = {
            'evaluation_split': split, 'n_examples': len(records), 'run_config_sha256': fingerprint,
            'source_csv_sha256': dataset_metadata['files'][split]['sha256'],
            'accuracy': float(accuracy_score(gold, predicted)),
            'macro_f1': float(f1_score(gold, predicted, labels=LABELS, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(gold, predicted, labels=LABELS, average='weighted', zero_division=0)),
            'invalid_rate': sum(p == 'INVALID' for p in predicted) / len(gold),
            'max_prompt_tokens': max_prompt_tokens, 'truncation': False,
            'elapsed_seconds': time.perf_counter() - started,
        }
        with (folder / 'predictions.jsonl').open('w', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record) + '\n')
        save_json(folder / 'classification_report.json', classification_report(
            gold, predicted, labels=LABELS, output_dict=True, zero_division=0))
        matrix_labels = LABELS + ['INVALID']
        pd.DataFrame(confusion_matrix(gold, predicted, labels=matrix_labels),
                     index=matrix_labels, columns=matrix_labels).loc[LABELS].to_csv(folder / 'confusion_matrix.csv')
        save_json(folder / 'metrics.json', metrics)
        return metrics
    finally:
        tokenizer.padding_side = previous_padding
        eval_model.train(previous_mode)
        gc.collect()
        torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class TAGCOSTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval', **kwargs):
        result = evaluate_labels(self.model, validation_frame,
                                 Path(self.args.output_dir) / 'validation' / f'step_{self.state.global_step}',
                                 'validation', self.run_fingerprint)
        metrics = {f'{metric_key_prefix}_{key}': result[key]
                   for key in ['accuracy', 'macro_f1', 'weighted_f1', 'invalid_rate']}
        metrics[f'{metric_key_prefix}_runtime'] = result['elapsed_seconds']
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

def run_subset(name):
    run_dir = OUTPUT_DIR / f'TinyLlama_TAGCOS_{name}_seed42'
    run_dir.mkdir(parents=True, exist_ok=True)
    train_data = TRAIN_SUBSETS[name]
    steps_per_epoch = math.ceil(math.ceil(len(train_data) / MICRO_BATCH) / GRAD_ACCUM)
    expected_steps = EPOCHS * steps_per_epoch
    assert expected_steps == {'50pct': 498, '25pct': 249, '10pct': 102, '5pct': 51}[name]
    run_config = {
        **baseline_config,
        'method': TAGCOS_CONFIG['method'], 'subset': name, 'clustering_seed': 42,
        'protocol': TAGCOS_CONFIG, 'protocol_sha256': PROTOCOL_HASH,
        'training_rows': len(train_data),
        'selection_sha256': sha256(SELECTION_DIR / f'subset_{name}.json'),
        'training_content_sha256': subset_content_hashes[name],
        'validation_content_sha256': provenance['verified_content_sha256']['validation'],
        'source_training_fingerprint': preprocessing['splits']['train']['dataset_fingerprint'],
        'initial_adapter_files_sha256': initial_hashes,
        'warmup_steps': math.ceil(0.03 * expected_steps), 'expected_optimizer_steps': expected_steps,
        'generation_config': generation_config.to_dict(), 'package_versions': package_versions,
        'source_full_run_config_sha256': json_hash(baseline_config),
    }
    # The full-data fingerprint is not the identity of a selected subset.
    run_config.pop('training_fingerprint', None)
    config_path = run_dir / 'run_config.json'
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, f'Configuration differs: {run_dir}'
    else:
        save_json(config_path, run_config)
    fingerprint = json_hash(run_config)
    completion_path = run_dir / 'training_complete.json'
    test_path = run_dir / 'test/metrics.json'
    complete = json.loads(completion_path.read_text()) if completion_path.exists() else None
    if complete is not None:
        assert complete['run_config_sha256'] == fingerprint
        assert sha256(run_dir / 'best_adapter/adapter_model.safetensors') == complete['best_adapter_sha256']
    if test_path.exists():
        assert complete is not None, 'Test metrics exist without a training completion record.'
        metrics = json.loads(test_path.read_text())
        assert metrics['run_config_sha256'] == fingerprint
        assert metrics['source_csv_sha256'] == dataset_metadata['files']['test']['sha256']
        assert metrics['n_examples'] == 1823
        for filename in ['predictions.jsonl', 'classification_report.json', 'confusion_matrix.csv']:
            assert (run_dir / 'test' / filename).is_file(), filename
        print(f'{name}: reusing completed training and test results.')
        return complete, metrics

    local_base = local_model = local_trainer = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        set_seed(SEED)
        tokenizer.padding_side = 'right'
        print(f'\nTinyLlama TAGCOS {name}: {len(train_data):,} training rows')
        local_base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
            attn_implementation='sdpa', device_map={'': 'cuda:0'},
        )
        adapter_path = run_dir / 'best_adapter' if complete is not None else INITIAL_ADAPTER
        local_model = PeftModel.from_pretrained(local_base, str(adapter_path), is_trainable=complete is None)
        local_model.config.use_cache = False
        local_model.config.pad_token_id = tokenizer.pad_token_id
        local_model.generation_config = generation_config
        local_model.gradient_checkpointing_disable()
        if complete is None:
            for parameter_name, parameter in local_model.named_parameters():
                if parameter.requires_grad:
                    assert 'lora_' in parameter_name
                    parameter.data = parameter.data.float()
            del parameter
            assert sum(p.numel() for p in local_model.parameters() if p.requires_grad) == 12615680
            args = SingleGPUArguments(
                output_dir=str(run_dir), num_train_epochs=EPOCHS,
                per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=baseline_config['learning_rate'], warmup_steps=run_config['warmup_steps'],
                lr_scheduler_type='linear', optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
                fp16=True, bf16=False, gradient_checkpointing=False,
                eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, save_only_model=False,
                load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
                logging_steps=25, logging_first_step=True, report_to='none',
                dataloader_num_workers=0, dataloader_pin_memory=True,
                seed=SEED, data_seed=SEED, label_names=['labels'],
            )
            assert args.world_size == 1 and args.device == DEVICE and args.train_batch_size == MICRO_BATCH
            (run_dir / 'training_args.json').write_text(args.to_json_string(), encoding='utf-8')
            local_trainer = TAGCOSTrainer(
                model=local_model, args=args, train_dataset=train_data, eval_dataset=full_validation,
                data_collator=training_collator, processing_class=tokenizer,
            )
            local_trainer.run_fingerprint = fingerprint
            checkpoint = get_last_checkpoint(str(run_dir))
            if checkpoint is not None:
                for filename in ['adapter_model.safetensors', 'adapter_config.json',
                                 'optimizer.pt', 'scheduler.pt', 'trainer_state.json',
                                 'rng_state.pth', 'scaler.pt']:
                    assert (Path(checkpoint) / filename).is_file(), (
                        f'Incomplete resume checkpoint: {checkpoint}/{filename}'
                    )
            print('Starting from:', checkpoint or INITIAL_ADAPTER)
            print('Expected optimizer steps:', expected_steps)
            torch.cuda.reset_peak_memory_stats(DEVICE)
            result = local_trainer.train(resume_from_checkpoint=checkpoint)
            assert local_trainer.state.global_step == expected_steps
            assert local_trainer.state.epoch >= EPOCHS - 1e-6
            assert local_trainer.state.best_metric is not None
            validation_scores = [entry['eval_macro_f1']
                                 for entry in local_trainer.state.log_history
                                 if 'eval_macro_f1' in entry]
            assert len(validation_scores) == EPOCHS
            assert abs(max(validation_scores) - local_trainer.state.best_metric) < 1e-12
            local_trainer.save_model(str(run_dir / 'best_adapter'))
            tokenizer.save_pretrained(str(run_dir / 'best_adapter'))
            local_trainer.save_metrics('train', result.metrics)
            local_trainer.save_state()
            complete = {
                'run_config_sha256': fingerprint,
                'best_validation_macro_f1': float(local_trainer.state.best_metric),
                'best_checkpoint': local_trainer.state.best_model_checkpoint,
                'global_step': int(local_trainer.state.global_step), 'epoch': float(local_trainer.state.epoch),
                'train_call_seconds': result.metrics['train_runtime'],
                'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
                'best_adapter_sha256': sha256(run_dir / 'best_adapter/adapter_model.safetensors'),
                'resumed_from': checkpoint, 'runtime_note': 'A resumed call excludes earlier sessions.',
            }
            save_json(completion_path, complete)
            local_model = local_trainer.model
        # Test data is read only after the validation-selected adapter is fixed.
        test_frame = load_original('test', 'healthver_test.csv')
        assert len(test_frame) == 1823
        assert set(test_frame['id']).isdisjoint(train_frame['id'])
        assert set(test_frame['id']).isdisjoint(validation_frame['id'])
        metrics = evaluate_labels(local_model, test_frame, run_dir / 'test', 'test', fingerprint)
        print(f"Best validation F1: {complete['best_validation_macro_f1']:.4f} | "
              f"Test F1: {metrics['macro_f1']:.4f} | Test accuracy: {metrics['accuracy']:.4f}")
        return complete, metrics
    finally:
        local_trainer = local_model = local_base = None
        gc.collect()
        torch.cuda.empty_cache()

summary_rows = []
for subset_name in ['50pct', '25pct', '10pct', '5pct']:
    completed, metrics = run_subset(subset_name)
    summary_rows.append({
        'model': MODEL_ID, 'method': TAGCOS_CONFIG['method'], 'subset': subset_name,
        'training_rows': SUBSET_SIZES[subset_name], 'clustering_seed': 42, 'training_seed': SEED,
        'best_validation_macro_f1': completed['best_validation_macro_f1'],
        'test_accuracy': metrics['accuracy'], 'test_macro_f1': metrics['macro_f1'],
        'test_weighted_f1': metrics['weighted_f1'], 'test_invalid_rate': metrics['invalid_rate'],
        'train_call_seconds': completed['train_call_seconds'], 'test_seconds': metrics['elapsed_seconds'],
    })
    summary = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / 'tagcos_results.csv'
    temporary = summary_path.with_suffix('.tmp.csv')
    summary.to_csv(temporary, index=False)
    temporary.replace(summary_path)
    print(summary.to_string(index=False))

print('\nAll four runs complete.')
print('Results:', OUTPUT_DIR / 'tagcos_results.csv')
print('Selections and training settings remained fixed throughout the loop.')



TinyLlama TAGCOS 50pct: 5,295 training rows


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Starting from: /kaggle/input/datasets/datasciencexlscout/base-3/HealthVer_TinyLlama_Experiments/initial_adapter_seed42
Expected optimizer steps: 498


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.184251,No log,0.636411,0.622722,0.651021,0.000000
2,0.106424,No log,0.758477,0.734268,0.764407,0.000000
3,0.068494,No log,0.779864,0.742650,0.777819,0.000000


validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

... (23 outputs left out)

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

TinyLlama_TAGCOS_5pct_seed42: test:   0%|          | 0/228 [00:00<?, ?it/s]

Best validation F1: 0.4371 | Test F1: 0.4251 | Test accuracy: 0.5041
                             model                      method subset  training_rows  clustering_seed  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  50pct           5295               42             42                  0.742650       0.748217       0.737819          0.746170                0.0           2252.0546    134.020841
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  25pct           2648               42             42                  0.650097       0.682940       0.654714          0.677253                0.0           1344.2155    133.828568
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  10pct           1059               42             42                  0.474127       0.545804       0.498886          0.532581         

In [12]:
results_path = TAGCOS_ROOT / 'tagcos_results.csv'
results = pd.read_csv(results_path)
assert results['subset'].tolist() == list(SUBSET_SIZES)
assert results['training_rows'].tolist() == list(SUBSET_SIZES.values())
assert (results['training_seed'] == 42).all()
test_frame = load_original('test', 'healthver_test.csv')
assert len(test_frame) == 1823
for name, size in SUBSET_SIZES.items():
    run = TAGCOS_ROOT / f'TinyLlama_TAGCOS_{name}_seed42'
    config = json.loads((run / 'run_config.json').read_text())
    complete = json.loads((run / 'training_complete.json').read_text())
    metrics = json.loads((run / 'test/metrics.json').read_text())
    fingerprint = json_hash(config)
    assert config['protocol_sha256'] == PROTOCOL_HASH
    assert config['selection_sha256'] == sha256(SELECTION_DIR / f'subset_{name}.json')
    assert config['initial_adapter_files_sha256'] == initial_hashes
    assert config['initial_adapter_sha256'] == WARMUP_MANIFEST['initial_adapter_sha256']
    assert complete['run_config_sha256'] == metrics['run_config_sha256'] == fingerprint
    assert complete['global_step'] == 3 * math.ceil(size / 32)
    assert complete['epoch'] >= 3 - 1e-6
    assert sha256(run / 'best_adapter/adapter_model.safetensors') == complete['best_adapter_sha256']
    assert metrics['n_examples'] == 1823 and metrics['evaluation_split'] == 'test'
    records = [json.loads(line) for line in (run / 'test/predictions.jsonl').read_text().splitlines()]
    assert [r['id'] for r in records] == test_frame['id'].tolist()
    assert [r['true_label'] for r in records] == test_frame['label'].tolist()
    assert all(r['predicted_label'] == parse_prediction(r['raw_response']) for r in records)
    reconstructed = f1_score([r['true_label'] for r in records], [r['predicted_label'] for r in records],
                             labels=LABELS, average='macro', zero_division=0)
    assert abs(reconstructed - metrics['macro_f1']) < 1e-12
    assert abs(reconstructed - float(results.loc[results['subset']==name, 'test_macro_f1'].iloc[0])) < 1e-12
    validation_metrics = [json.loads(p.read_text()) for p in (run/'validation').glob('step_*/metrics.json')]
    assert len(validation_metrics) == 3
    assert all(m['run_config_sha256'] == fingerprint for m in validation_metrics)
    assert abs(max(m['macro_f1'] for m in validation_metrics) - complete['best_validation_macro_f1']) < 1e-12

feature_seconds = sum(json.loads((TAGCOS_ROOT/'features'/f'checkpoint-{s}'/'feature_metadata.json').read_text())['extraction_seconds']
                      for s in CHECKPOINT_STEPS)
warmup_seconds = float(json.loads((WARMUP_DIR / 'train_results.json').read_text())['train_runtime'])
aggregation_seconds = json.loads(aggregate_meta_path.read_text())['aggregation_seconds']
clustering_seconds = json.loads(cluster_meta_file.read_text())['clustering_seconds']
omp_seconds = {name:json.loads((SELECTION_DIR/f'subset_{name}.json').read_text())['selection_seconds']
               for name in SUBSET_SIZES}
shared_seconds = warmup_seconds + feature_seconds + aggregation_seconds + clustering_seconds
cost_report = {
    'protocol_sha256':PROTOCOL_HASH, 'warmup_seconds':warmup_seconds,
    'feature_extraction_seconds':feature_seconds, 'aggregation_seconds':aggregation_seconds,
    'clustering_seconds':clustering_seconds, 'omp_seconds_by_subset':omp_seconds,
    'selection_seconds_by_subset_including_shared_work':{name:shared_seconds+t for name,t in omp_seconds.items()},
    'all_budgets_selection_seconds_with_shared_work_counted_once':shared_seconds+sum(omp_seconds.values()),
    'timing_note':'Includes warmup, feature extraction, aggregation, clustering and OMP. '
                  'Feature extraction includes its inline sanity check. Excludes model loading, package/setup checks and unsaved interrupted work. '
                  'Final train-call times include validation; resumed calls omit earlier sessions. '
                  'These are recorded runtimes, not precise total GPU-hours.',
}
save_json(TAGCOS_ROOT / 'selection_costs.json', cost_report)
save_json(TAGCOS_ROOT / 'final_audit.json', {
    'status':'PASSED', 'protocol_sha256':PROTOCOL_HASH,
    'results_sha256':sha256(results_path), 'checked_subsets':list(SUBSET_SIZES),
    'test_rows':1823, 'training_seed':42,
    'claim_scope':'One-seed artifact consistency; no statistical superiority claim.',
})
print(results.to_string(index=False))
print('\nRecorded selection minutes (shared work counted once):',
      round(cost_report['all_budgets_selection_seconds_with_shared_work_counted_once']/60, 2))
print('FINAL ARTIFACT AUDIT PASSED.')
print('Results:', results_path)
print('Audit:', TAGCOS_ROOT / 'final_audit.json')
print('Selection costs:', TAGCOS_ROOT / 'selection_costs.json')


                             model                      method subset  training_rows  clustering_seed  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  50pct           5295               42             42                  0.742650       0.748217       0.737819          0.746170                0.0           2252.0546    134.020841
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  25pct           2648               42             42                  0.650097       0.682940       0.654714          0.677253                0.0           1344.2155    133.828568
TinyLlama/TinyLlama-1.1B-Chat-v1.0 TAGCOS_HealthVer_adaptation  10pct           1059               42             42                  0.474127       0.545804       0.498886          0.532581                0.0            792.8369    133.019326
TinyLlama/TinyLlama-1.1B